# 02 — Few-shot Node Classification (Tables 2 & 3)

Same 8 datasets and leave-one-dataset-out protocol as `01_node_classification.ipynb`, but with `--shots 3` and `--shots 5` (Table 2 = 3-shot, Table 3 = 5-shot). `--shots` controls k in the k-shot fine-tuning split (`trainers/node2graph_trainer.py::one_shot_finetune`), independent of pretraining.

To keep this notebook a manageable size (8 datasets x 2 shot settings = 16 full pretrain+finetune runs), each shot setting runs as one loop cell rather than one cell per dataset. If a run in the middle of a loop fails, just re-run that loop — completed datasets are fast to skip by checking the log-existence guard below, but note pretraining itself is **not** cached across shot settings (each `main.py` invocation retrains from scratch), so re-running is expensive; consider commenting out already-done datasets from the list if you need to resume.

**Prereq.** Run `00_setup.ipynb` in this same Colab session first.

In [ ]:
import os
assert 'REPO_DIR' in globals(), 'Run 00_setup.ipynb first in this session.'
os.chdir(REPO_DIR)
DATASETS = ["wisconsin", "texas", "cornell", "citeseer", "cora", "pubmed", "computers", "photo"]
print(DATASETS)

In [ ]:
# 3-shot (Table 2)
FEWSHOT3_LOG_DIR = f'{RESULTS_DIR}/nc_3shot'
os.makedirs(FEWSHOT3_LOG_DIR, exist_ok=True)

for ds in DATASETS:
    log = f'{FEWSHOT3_LOG_DIR}/{ds}.log'
    if os.path.exists(log):
        print(f'--- {ds}: log already exists, skipping (delete the log to force a re-run) ---')
        continue
    print(f'=== 3-shot: {ds} ===')
    !python main.py --dataset {ds} --epochs 150 --shots 3 --device 0 \
        --dataset_dir "$DATA_DIR" --checkpoint_dir "$CKPT_DIR" --checkpoint_prefix node2graph_3shot_{ds} \
        2>&1 | tee $log

In [ ]:
# 5-shot (Table 3)
FEWSHOT5_LOG_DIR = f'{RESULTS_DIR}/nc_5shot'
os.makedirs(FEWSHOT5_LOG_DIR, exist_ok=True)

for ds in DATASETS:
    log = f'{FEWSHOT5_LOG_DIR}/{ds}.log'
    if os.path.exists(log):
        print(f'--- {ds}: log already exists, skipping (delete the log to force a re-run) ---')
        continue
    print(f'=== 5-shot: {ds} ===')
    !python main.py --dataset {ds} --epochs 150 --shots 5 --device 0 \
        --dataset_dir "$DATA_DIR" --checkpoint_dir "$CKPT_DIR" --checkpoint_prefix node2graph_5shot_{ds} \
        2>&1 | tee $log

In [ ]:
# Summary for both shot settings.
import re

def summarize(log_dir, datasets):
    for ds in datasets:
        log = f'{log_dir}/{ds}.log'
        if not os.path.exists(log):
            print(f'{ds:12s}  (not run yet)')
            continue
        txt = open(log).read()
        m = re.findall(r'Test Accuracy:\s*([0-9.]+)\s*\+/-\s*([0-9.]+)', txt)
        if m:
            acc, std = m[-1]
            print(f'{ds:12s}  {float(acc)*100:.2f} +/- {float(std)*100:.2f}')
        else:
            print(f'{ds:12s}  (no result line found — check log)')

print('--- 3-shot ---')
summarize(FEWSHOT3_LOG_DIR, DATASETS)
print('\n--- 5-shot ---')
summarize(FEWSHOT5_LOG_DIR, DATASETS)

**Target numbers (paper, R-GFM row, accuracy %).**

3-shot (Table 2):

| Wisconsin | Cornell | Citeseer | Cora | Pubmed | Computers | Photos | Texas |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 43.10 ± 4.03 | 45.39 ± 3.09 | 73.98 ± 1.42 | 59.26 ± 5.04 | 59.19 ± 3.01 | 56.02 ± 3.90 | 71.50 ± 2.80 | 44.18 ± 7.02 |

5-shot (Table 3):

| Wisconsin | Cornell | Citeseer | Cora | Pubmed | Computers | Photos | Texas |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 47.75 ± 4.97 | 47.97 ± 2.82 | 74.59 ± 1.87 | 63.77 ± 2.47 | 63.39 ± 1.85 | 59.55 ± 4.17 | 73.62 ± 1.15 | 51.64 ± 6.31 |